# Dataset Collection & Processing

This notebook covers:
1. Dataset conversion (Excel → JSON)
2. LLM augmentation for detailed reasoning
3. Dataset merging and balancing
4. Format conversion for fine-tuning

## 1. Dataset Conversion

In [ ]:
import sys
sys.path.append('../tools')

from dataset_converter import DatasetConverter
import pandas as pd
import json

### Load and Inspect Dataset

In [ ]:
# Load dataset
df = pd.read_excel('../native_ads_dataset.xlsx', sheet_name='Clean')

print(f"Total samples: {len(df)}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())

# Show sample
df.head()

### Convert to Instruction Format

In [ ]:
# Initialize converter
converter = DatasetConverter('../native_ads_dataset.xlsx')

# Convert to instruction format
instruction_data = converter.convert_to_instruction_format(
    content_column='content',
    label_column='label'
)

# Save
converter.save_to_json(instruction_data, '../data/llm_dataset_instruction.json')

print(f"✅ Converted {len(instruction_data)} samples")

## 2. LLM Augmentation (Optional)

Generate detailed reasoning using GPT-4o-mini

In [ ]:
# This will take ~2-3 hours for full dataset
# Recommended: Use --max-samples for testing

!python ../tools/augment_with_openrouter.py --max-samples 100

## 3. Merge Datasets

In [ ]:
# Merge instruction dataset with detailed reasoning dataset
!python ../tools/merge_datasets.py \
  --inputs ../data/llm_dataset_instruction.json ../data/llm_dataset_detailed.json \
  --output ../data/llm_dataset_mixed.json

## 4. Convert Format for Fine-Tuning

In [ ]:
# Convert to JSON format matching inference prompt
!python ../tools/fix_dataset_format.py \
  --input ../data/llm_dataset_mixed.json \
  --output ../data/llm_dataset_mixed_json.json

## 5. Analyze Final Dataset

In [ ]:
# Load final dataset
with open('../data/llm_dataset_mixed_json.json', 'r') as f:
    final_data = json.load(f)

# Statistics
total = len(final_data)
native_count = sum(1 for d in final_data if 'native ads' in d['output'].lower())
berita_count = total - native_count

print(f"📊 Final Dataset Statistics:")
print(f"   Total samples: {total}")
print(f"   Native Ads: {native_count} ({native_count/total*100:.1f}%)")
print(f"   Berita Murni: {berita_count} ({berita_count/total*100:.1f}%)")

# Sample output
print(f"\n📝 Sample Output:")
print(json.dumps(json.loads(final_data[0]['output']), indent=2, ensure_ascii=False))

## Next Steps

Dataset is ready for fine-tuning! See `02_finetuning.ipynb`